In [0]:
%run "../../commons/commons_imports"

In [0]:
STREAM_PATH = f"{RAW_PATH}/STREAMING"
CHECKPOINT_PATH = (
    f"{S3_BASE_PATH}/checkpoints/stream_novos_resultados"
)
BRONZE_STREAM_PATH = f"{BRONZE_PATH}/STREAM_NOVOS_RESULTADOS"

In [0]:
schema = StructType([
    StructField("ANO_REFERENCIA", IntegerType()),
    StructField("CO_MUNICIPIO", IntegerType()),
    StructField("ID_TIPO_REDE", IntegerType()),
    StructField("PC_ALUNO_ALFABETIZADO", DoubleType()),
    StructField("VL_MEDIA_LP", DoubleType())
])

In [0]:
# =====================================================
# Leitura Streaming (Auto Loader)
# =====================================================

df_stream = (
    spark.readStream
        .format("cloudFiles")
        .option(
            "cloudFiles.format",
            "csv"
        )
        .option(
            "header",
            "true"
        )
        .schema(schema)
        .load(STREAM_PATH)
)

In [0]:
# =====================================================
# Escrita Streaming para Bronze (Delta)
# =====================================================

query = (

    df_stream

    .writeStream

    .format("delta")

    .outputMode("append")

    .option(
        "checkpointLocation",
        CHECKPOINT_PATH
    )

    .start(
        BRONZE_STREAM_PATH
    )

)

In [0]:
# =====================================================
# Mantém o Streaming Ativo
# =====================================================

query.awaitTermination()